# CLMBR-T landmark inference (weekly cutoffs, week 0 to 37 inclusive)
For each cutoff `k`, cutoff time = `episode_start + k weeks`, where `episode_start = min(inferred, recorded)`.
Each patient's full MEDS history is run through the model **once** (after a causality check); the embedding at each cutoff is the representation at the last token with `time <= cutoff`.

Tables written per cutoff `kk` (zero-padded):
- global: `{prefix}_wk{kk}` -> `patient_id, emb_0..emb_767, sequence_length` (patients with no event at/before the cutoff are omitted)
- local raw (optional): `{prefix}_trimester_wk{kk}` -> `patient_id, window_name, emb_*, n_tokens, unavailable, empty`
- local delta: `{prefix}_delta_wk{kk}` -> same columns, deltas via the `n_back=3` no-ramp-up logic

In [0]:
%pip install --force-reinstall --no-cache-dir torch==2.1.2 femr==0.2.3 datasets==2.15.0 xformers==0.0.23.post1 transformers==4.35.2
dbutils.library.restartPython()

In [0]:
%pip install --force-reinstall --no-cache-dir "urllib3<2"
dbutils.library.restartPython()

In [0]:
from huggingface_hub import login

hf_token = dbutils.secrets.get(scope="hf-secrets", key="hf_token")
login(token=hf_token)

In [0]:
import sys
sys.modules['flash_attn'] = None
sys.modules['flash_attn_2_cuda'] = None

import femr.models.transformer
import femr.models.tokenizer
import femr.models.processor
import femr.models.processor as femr_processor_module
import femr.models.xformers as xformers_module
import torch
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.compute as pc
import datasets
import gc
import time
from itertools import groupby
from operator import itemgetter

# ============================================================================
# Config -- all hyperparameters centralized here
# ============================================================================
CFG = {
    "model_name": "StanfordShahLab/clmbr-t-base",
    "meds_path": "/path/to/exports/meds_output_spark/data",
    "cohort_path": "catalog.schema.step7_filtered_cohort_with_apos",

    # all output tables hang off this prefix (see markdown cell above)
    "table_prefix": "catalog.schema.patient_clmbr_embeddings_landmark",

    # landmark cutoffs: weeks since episode_start, inclusive on both ends
    "cutoff_week_start": 0,
    "cutoff_week_end": 37,

    "sample_size": None,
    "write_chunk_size": 5000,
    "embed_dim": 768,                 # clmbr-t-base hidden size

    # local branch: same rolling-trimester grid as the original extraction
    "tri_roll_pre_months": 6,
    "tri_roll_post_months": 13,
    "tri_roll_stride_months": 1,
    "tri_roll_window_months": 3,
    "delta_n_back": 3,                # window[i-3] end == window[i] start exactly
    "save_raw_local": False,           # False = write only delta tables (saves ~half the local storage)

    # causality check (one-pass inference is only valid if the model is causal)
    "check_n_patients": 10,
    "check_atol": 1e-3,
}

CUTOFF_WEEKS = np.arange(CFG["cutoff_week_start"], CFG["cutoff_week_end"] + 1)
EMB_COLS = [f"emb_{d}" for d in range(CFG["embed_dim"])]

def tbl_global(k):  return f"{CFG['table_prefix']}_wk{k:02d}"
def tbl_local(k):   return f"{CFG['table_prefix']}_trimester_wk{k:02d}"
def tbl_delta(k):   return f"{CFG['table_prefix']}_delta_wk{k:02d}"
TBL_EXCLUDED = f"{CFG['table_prefix']}_excluded"
TBL_PROGRESS = f"{CFG['table_prefix']}_progress"

if CFG["sample_size"] is not None:
    CFG["table_prefix"] += "_sample"
    TBL_EXCLUDED = f"{CFG['table_prefix']}_excluded"
    TBL_PROGRESS = f"{CFG['table_prefix']}_progress"
    print(f"sample_size={CFG['sample_size']} -- writing to *_sample tables")

if not torch.cuda.is_available():
    raise RuntimeError("No GPU available -- stopping rather than silently running on CPU.")
device = "cuda"
print(f"torch={torch.__version__}, cuda={torch.version.cuda}, GPU={torch.cuda.get_device_name(0)}")
print(f"{len(CUTOFF_WEEKS)} cutoffs: week {CUTOFF_WEEKS[0]}..{CUTOFF_WEEKS[-1]}")


# ============================================================================
# Window boundaries (vectorized across the cohort) -- identical to the original
# ============================================================================
def precompute_rolling_trimester_boundaries(starts, pre_months, post_months, stride_months, window_months):
    names, wstart_cols, wend_cols = [], [], []
    for offset in range(-pre_months, post_months, stride_months):
        names.append(f"tri_mo{offset:+03d}")
        wstart = starts + pd.DateOffset(months=offset)
        wend = wstart + pd.DateOffset(months=window_months)
        wstart_cols.append(wstart.values)
        wend_cols.append(wend.values)
    return names, np.column_stack(wstart_cols), np.column_stack(wend_cols)


def extract_windows_vectorized(times_sorted, reps_sorted, wstarts, wends, embed_dim):
    """Unchanged from the original. Raw embedding of window i = representation at the
    last token with time <= wend[i]; this does not depend on the landmark cutoff
    because the model is causal (verified by the check cell)."""
    n_windows = len(wstarts)
    if len(times_sorted) == 0:
        return (np.full((n_windows, embed_dim), np.nan, dtype=np.float32),
                np.full(n_windows, np.nan),
                np.ones(n_windows, dtype=np.int64),
                np.zeros(n_windows, dtype=np.int64))

    last_event_time = times_sorted[-1]
    first_event_time = times_sorted[0]
    wstarts = wstarts.astype(times_sorted.dtype)
    wends = wends.astype(times_sorted.dtype)

    after_cutoff = wends > last_event_time
    before_history = wends < first_event_time
    available = ~after_cutoff & ~before_history

    idx = np.searchsorted(times_sorted, wends, side="right") - 1
    idx_clipped = np.clip(idx, 0, len(times_sorted) - 1)
    emb_matrix = reps_sorted[idx_clipped].astype(np.float32, copy=True)
    emb_matrix[~available] = np.nan

    lo = np.searchsorted(times_sorted, wstarts, side="left")
    hi = np.searchsorted(times_sorted, wends, side="left")
    n_tokens = (hi - lo).astype(np.float64)
    n_tokens[after_cutoff] = np.nan

    unavailable = (~available).astype(np.int64)
    empty = ((n_tokens == 0) & available).astype(np.int64)
    return emb_matrix, n_tokens, unavailable, empty


def build_long_df_from_arrays(pids_repeated, names_tiled, emb_matrix, n_tokens, unavailable, empty):
    df = pd.DataFrame(emb_matrix, columns=EMB_COLS)
    df.insert(0, "window_name", names_tiled)
    df.insert(0, "patient_id", pids_repeated)
    df["n_tokens"] = n_tokens
    df["unavailable"] = unavailable
    df["empty"] = empty
    return df


def process_batch_rolling_delta_no_rampup(batch_df, batch_label, window_order, n_back, embed_dim, verbose=False):
    """Same logic as the original delta script (first available window = raw cumulative
    base row; n_back-th available window onward = true n_back-step delta; windows in
    between stay NaN). Only change: `verbose` gates the per-batch ramp-up message so
    38 cutoffs x many chunks don't flood the output."""
    n_windows = len(window_order)

    if batch_df.duplicated(subset=["patient_id", "window_name"]).any():
        raise ValueError(f"{batch_label}: duplicate (patient_id, window_name) rows.")
    counts = batch_df.groupby("patient_id").size()
    if (counts != n_windows).any():
        raise ValueError(f"{batch_label}: some patients don't have exactly {n_windows} rows.")
    unknown_names = set(batch_df["window_name"].unique()) - set(window_order)
    if unknown_names:
        raise ValueError(f"{batch_label}: unrecognized window_name values: {unknown_names}.")

    batch_df = batch_df.copy()
    batch_df["window_name"] = pd.Categorical(batch_df["window_name"], categories=window_order, ordered=True)
    batch_df = batch_df.sort_values(["patient_id", "window_name"]).reset_index(drop=True)

    n_patients = batch_df["patient_id"].nunique()
    patient_order = batch_df["patient_id"].drop_duplicates().tolist()

    emb_matrix = batch_df[EMB_COLS].values.reshape(n_patients, n_windows, embed_dim)
    unavailable_matrix = batch_df["unavailable"].values.reshape(n_patients, n_windows).astype(int)

    # invariant: unavailable windows form only a leading and/or trailing block
    avail_indicator = (unavailable_matrix == 0).astype(int)
    padded = np.pad(avail_indicator, ((0, 0), (1, 1)), constant_values=0)
    n_available_runs = (np.diff(padded, axis=1) == 1).sum(axis=1)
    n_violations = int((n_available_runs > 1).sum())
    if n_violations > 0:
        ex = np.array(patient_order)[n_available_runs > 1][:5]
        print(f"  WARNING ({batch_label}): {n_violations} patients have a non-contiguous "
              f"availability pattern. Example patient_ids: {ex.tolist()}.")

    has_any_available = (unavailable_matrix == 0).any(axis=1)
    first_avail_idx = np.argmax(unavailable_matrix == 0, axis=1)

    delta_matrix = np.full_like(emb_matrix, np.nan, dtype=np.float32)
    if n_back < n_windows:
        delta_matrix[:, n_back:, :] = emb_matrix[:, n_back:, :] - emb_matrix[:, :-n_back, :]

    valid_patients = np.where(has_any_available)[0]
    delta_matrix[valid_patients, first_avail_idx[valid_patients], :] = \
        emb_matrix[valid_patients, first_avail_idx[valid_patients], :]

    out_df = batch_df.copy()
    out_df[EMB_COLS] = delta_matrix.reshape(n_patients * n_windows, embed_dim)
    out_df["window_name"] = out_df["window_name"].astype(str)

    if verbose:
        n_base = int(has_any_available.sum())
        n_delta = int(np.isfinite(delta_matrix).all(axis=2).sum()) - n_base
        n_dropped = int((unavailable_matrix == 0).sum()) - n_base - n_delta
        print(f"  {batch_label}: {n_base} base, {n_delta} delta, {n_dropped} ramp-up windows left NaN")
    return out_df


# ----------------------------------------------------------------------------
# Batching / inference helpers
# ----------------------------------------------------------------------------
def make_batch(batch_processor, patients, tensor_type="pt"):
    # no max_seq_len here: truncating history would change what each cutoff embedding sees
    creator = batch_processor.creator
    creator.start_batch()
    for p in patients:
        creator.add_patient(p)
    batch_data = creator.get_batch_data()
    if tensor_type is not None:
        formatter = datasets.formatting.get_formatter(tensor_type)
        batch_data = formatter.recursive_tensorize(batch_data)
    batch_data = creator.cleanup_batch(batch_data)
    return {"batch": femr_processor_module._add_dimension(batch_data)}


def move_to_device(obj, device):
    if torch.is_tensor(obj):
        return obj.to(device)
    elif isinstance(obj, dict):
        return {k: move_to_device(v, device) for k, v in obj.items()}
    return obj


def get_batch_size(seq_len, target_tokens=4096, max_batch_size=64):
    return max(1, min(max_batch_size, target_tokens // max(seq_len, 1)))


def load_chunk_patients(dataset, chunk_ids):
    """Loads one chunk's MEDS rows (ALL history, including pre-episode data) and builds
    patient dicts plus per-patient unique event times (for sequence_length)."""
    filter_expr = pc.field('subject_id').isin(chunk_ids)
    df_chunk = dataset.to_table(columns=['subject_id', 'time', 'code'], filter=filter_expr).to_pandas()
    df_chunk = df_chunk.sort_values(['subject_id', 'time'])
    sub_list = df_chunk['subject_id'].to_numpy().tolist()
    time_list = df_chunk['time'].to_numpy().astype('datetime64[us]').tolist()
    code_list = df_chunk['code'].to_numpy().tolist()
    del df_chunk

    patients, n_events, event_times = [], {}, {}
    for subject_id, patient_rows in groupby(zip(sub_list, time_list, code_list), key=itemgetter(0)):
        events = []
        for t, time_rows in groupby(patient_rows, key=itemgetter(1)):
            events.append({'time': t, 'measurements': [{'code': c} for _, _, c in time_rows]})
        patients.append({'patient_id': subject_id, 'events': events})
        n_events[subject_id] = len(events)
        event_times[subject_id] = np.array([e['time'] for e in events], dtype='datetime64[s]')
    patients_sorted = sorted(patients, key=lambda p: n_events[p['patient_id']])
    return patients_sorted, n_events, event_times


def iter_patient_tokens(patients_sorted, n_events):
    """Yields (patient_id, times_sorted[s], reps_sorted) for every patient, batching by length."""
    i, n = 0, len(patients_sorted)
    while i < n:
        probe_len = n_events[patients_sorted[min(i + 15, n - 1)]['patient_id']]
        batch_patients = patients_sorted[i:i + get_batch_size(probe_len)]
        batch = move_to_device(make_batch(batch_processor, batch_patients), device)
        with torch.no_grad():
            _, result = model(**batch)
        pids = result['patient_ids'].cpu().numpy()
        times = result['timestamps'].cpu().numpy().astype('datetime64[s]')
        reps = result['representations'].cpu().numpy()

        order = np.lexsort((times, pids))
        pids_o, times_o, reps_o = pids[order], times[order], reps[order]
        uniq, starts = np.unique(pids_o, return_index=True)   # pids_o is sorted -> first occurrences
        ends = np.append(starts[1:], len(pids_o))
        for pid, s, e in zip(uniq, starts, ends):
            yield int(pid), times_o[s:e], reps_o[s:e]
        i += len(batch_patients)


# ----------------------------------------------------------------------------
# Verify real attn_bias on the live forward pass (idempotent across reruns)
# ----------------------------------------------------------------------------
if not hasattr(xformers_module, '_true_original_wrapper'):
    xformers_module._true_original_wrapper = xformers_module.memory_efficient_attention_wrapper
_true_original_wrapper = xformers_module._true_original_wrapper
_checked = {'done': False}

def _wrapper_with_check(q, k, v, attn_bias):
    if not _checked['done']:
        assert q.device.type == "cuda", f"Expected cuda tensor, got {q.device.type}"
        assert attn_bias is not None, "attn_bias is None -- patch is still active somewhere!"
        print(f"VERIFIED: live forward pass using real xformers CUDA kernel with "
              f"real {type(attn_bias).__name__} (not CPU fallback, not disabled masking).")
        _checked['done'] = True
    return _true_original_wrapper(q, k, v, attn_bias)

xformers_module.memory_efficient_attention_wrapper = _wrapper_with_check

In [0]:
# ----------------------------------------------------------------------------
# Cohort, episode starts, window grids, cutoff times, usable ids, model
# ----------------------------------------------------------------------------
t0 = time.time()
cohort_df = spark.table(CFG["cohort_path"]).select(
    "person_id", "recorded_episode_start", "inferred_episode_start").toPandas()
cohort_df["recorded_episode_start"] = pd.to_datetime(cohort_df["recorded_episode_start"])
cohort_df["inferred_episode_start"] = pd.to_datetime(cohort_df["inferred_episode_start"])
# earliest of the two (skipna: falls back to whichever exists)
cohort_df["episode_start"] = cohort_df[["recorded_episode_start", "inferred_episode_start"]].min(axis=1, skipna=True)

n_no_start = int(cohort_df["episode_start"].isna().sum())
if n_no_start > 0:
    print(f"WARNING: {n_no_start} patients have no episode_start -- excluded (cannot anchor cutoffs).")
cohort_df = cohort_df.dropna(subset=["episode_start"]).reset_index(drop=True)
pid_to_row = {pid: i for i, pid in enumerate(cohort_df["person_id"].values)}

# cutoff times: (n_patients, n_cutoffs), datetime64[s]
starts_s = cohort_df["episode_start"].values.astype("datetime64[s]")
CUT_ALL = starts_s[:, None] + (CUTOFF_WEEKS * 7).astype("timedelta64[D]")[None, :]

TRI_NAMES, TRI_WSTART_ALL, TRI_WEND_ALL = precompute_rolling_trimester_boundaries(
    cohort_df["episode_start"],
    pre_months=CFG["tri_roll_pre_months"], post_months=CFG["tri_roll_post_months"],
    stride_months=CFG["tri_roll_stride_months"], window_months=CFG["tri_roll_window_months"])
TRI_WEND_ALL_S = TRI_WEND_ALL.astype("datetime64[s]")
N_WIN = len(TRI_NAMES)
print(f"[timing] cohort + boundaries: {time.time()-t0:.1f}s | {len(pid_to_row)} patients with episode_start, "
      f"{N_WIN} trimester windows, {len(CUTOFF_WEEKS)} cutoffs")

# usable patients: have clinical codes (not just a 'person' row) AND a resolvable episode start
dataset = ds.dataset(CFG["meds_path"], format="parquet")
id_table_df = dataset.to_table(columns=['subject_id', 'table']).to_pandas()
table_composition = id_table_df.groupby('subject_id')['table'].apply(set)
no_clinical_codes_ids = set(table_composition[table_composition.apply(lambda t: t == {'person'})].index)
usable_ids = sorted(set(id_table_df['subject_id'].unique()) - no_clinical_codes_ids)
del id_table_df
usable_ids = [pid for pid in usable_ids if pid in pid_to_row]
if CFG["sample_size"] is not None:
    usable_ids = usable_ids[:CFG["sample_size"]]
print(f"Usable patients: {len(usable_ids)}")

tokenizer = femr.models.tokenizer.FEMRTokenizer.from_pretrained(CFG["model_name"])
batch_processor = femr.models.processor.FEMRBatchProcessor(tokenizer)
model = femr.models.transformer.FEMRModel.from_pretrained(CFG["model_name"])
model.eval()
model.to(device)
torch.cuda.reset_peak_memory_stats()
print(f"Model loaded. allocated={torch.cuda.memory_allocated()/1e6:.1f} MB")

In [0]:
# ----------------------------------------------------------------------------
# CAUSALITY CHECK: one-pass inference is valid only if the representation at time T
# from the FULL history equals the one from history TRUNCATED at T. Stops the run if not.
# ----------------------------------------------------------------------------
def last_rep_at_or_before(patient, cutoff_dt):
    batch = move_to_device(make_batch(batch_processor, [patient]), device)
    with torch.no_grad():
        _, r = model(**batch)
    t = r['timestamps'].cpu().numpy().astype('datetime64[s]')
    reps = r['representations'].cpu().numpy()
    order = np.argsort(t, kind="stable")
    t, reps = t[order], reps[order]
    idx = np.searchsorted(t, np.datetime64(cutoff_dt, 's'), side="right") - 1
    return reps[idx]

check_patients, _, _ = load_chunk_patients(dataset, usable_ids[:CFG["check_n_patients"] * 3])
check_patients = [p for p in check_patients if len(p['events']) >= 4][:CFG["check_n_patients"]]
assert len(check_patients) > 0, "no patients with >=4 events available for the causality check"

max_diffs = []
for p in check_patients:
    mid = len(p['events']) // 2
    cut = p['events'][mid]['time']
    full = last_rep_at_or_before(p, cut)
    trunc = last_rep_at_or_before({'patient_id': p['patient_id'], 'events': p['events'][:mid + 1]}, cut)
    max_diffs.append(float(np.abs(full - trunc).max()))
print(f"causality check on {len(max_diffs)} patients: max |full - truncated| = {max(max_diffs):.2e} "
      f"(atol={CFG['check_atol']})")
if max(max_diffs) > CFG["check_atol"]:
    raise RuntimeError("Model is NOT causal under this pipeline -- one-pass inference would leak future "
                       "information into earlier cutoffs. Switch to truncate-and-rerun per cutoff.")
print("PASSED: one-pass inference is valid.")
del check_patients

In [0]:
# ----------------------------------------------------------------------------
# Main loop: chunks of patients -> one inference pass -> write every cutoff's tables.
# Resume is chunk-level via a progress table; a chunk that started but did not finish
# has its rows deleted from all output tables before it is reprocessed (no duplicates).
# Requires the same usable_ids and write_chunk_size as the earlier run.
# ----------------------------------------------------------------------------
def write_append(df, table):
    spark.createDataFrame(df).write.format("delta").mode("append").saveAsTable(table)

def log_progress(chunk_idx, lo, hi, status):
    write_append(pd.DataFrame({"chunk_idx": [chunk_idx], "first_id": [int(lo)], "last_id": [int(hi)],
                               "status": [status]}), TBL_PROGRESS)

output_tables = ([tbl_global(k) for k in CUTOFF_WEEKS] + [tbl_delta(k) for k in CUTOFF_WEEKS]
                 + ([tbl_local(k) for k in CUTOFF_WEEKS] if CFG["save_raw_local"] else []))

try:
    prog = spark.table(TBL_PROGRESS).toPandas()
    done_chunks = set(prog.loc[prog["status"] == "done", "chunk_idx"])
    started_chunks = set(prog.loc[prog["status"] == "started", "chunk_idx"])
except Exception:
    done_chunks, started_chunks = set(), set()
print(f"Resume: {len(done_chunks)} chunks already done")

D = CFG["embed_dim"]
n_chunks = (len(usable_ids) + CFG["write_chunk_size"] - 1) // CFG["write_chunk_size"]
t_overall = time.time()
total_done = 0

for chunk_idx in range(n_chunks):
    if chunk_idx in done_chunks:
        continue
    chunk_ids = usable_ids[chunk_idx * CFG["write_chunk_size"]:(chunk_idx + 1) * CFG["write_chunk_size"]]
    lo, hi = chunk_ids[0], chunk_ids[-1]    # usable_ids is sorted, so [lo, hi] holds exactly this chunk's ids
    t_chunk = time.time()

    if chunk_idx in started_chunks:         # crashed mid-chunk last time: purge partial rows
        for t in output_tables:
            if spark.catalog.tableExists(t):
                spark.sql(f"DELETE FROM {t} WHERE patient_id BETWEEN {int(lo)} AND {int(hi)}")
        print(f"  chunk {chunk_idx}: purged partial rows from a previous incomplete run")
    log_progress(chunk_idx, lo, hi, "started")

    patients_sorted, n_events, event_times = load_chunk_patients(dataset, chunk_ids)

    # ---- one inference pass; per patient, derive all cutoffs' global + local arrays ----
    pids_l, g_emb, g_valid, g_seq = [], [], [], []
    l_emb, l_nt, l_un, l_wend = [], [], [], []
    for pid, p_times, p_reps in iter_patient_tokens(patients_sorted, n_events):
        row = pid_to_row[pid]
        cuts = CUT_ALL[row]                                              # (n_cutoffs,)

        # global: last token with time <= cutoff (vectorized over cutoffs)
        idx = np.searchsorted(p_times, cuts, side="right") - 1
        g_valid.append(idx >= 0)
        g_emb.append(p_reps[np.clip(idx, 0, None)].astype(np.float32))
        g_seq.append(np.searchsorted(event_times[pid], cuts, side="right"))   # unique event times <= cutoff

        # local: base per-window arrays (independent of cutoff); cutoff mask applied below
        e, nt, un, _ = extract_windows_vectorized(p_times, p_reps, TRI_WSTART_ALL[row], TRI_WEND_ALL[row], D)
        l_emb.append(e); l_nt.append(nt); l_un.append(un); l_wend.append(TRI_WEND_ALL_S[row])
        pids_l.append(pid)

    PIDS = np.array(pids_l)
    ROWS = np.array([pid_to_row[p] for p in pids_l])
    G_EMB, G_VALID, G_SEQ = np.stack(g_emb), np.stack(g_valid), np.stack(g_seq)         # (P,K,D) (P,K) (P,K)
    L_EMB, L_NT, L_UN, L_WEND = np.stack(l_emb), np.stack(l_nt), np.stack(l_un), np.stack(l_wend)  # (P,W,D) (P,W)
    del g_emb, g_valid, g_seq, l_emb, l_nt, l_un, l_wend, patients_sorted
    P = len(PIDS)

    for j, k in enumerate(CUTOFF_WEEKS):
        # ---- global table for cutoff k ----
        m = G_VALID[:, j]
        g_df = pd.DataFrame(G_EMB[m, j, :], columns=EMB_COLS)
        g_df.insert(0, "patient_id", PIDS[m])
        g_df["sequence_length"] = G_SEQ[m, j]
        write_append(g_df, tbl_global(k))

        # ---- local: a window is available at cutoff k only if it was available in the full
        # history AND it ends at/before the cutoff; later windows become NaN rows ----
        late = L_WEND > CUT_ALL[ROWS, j][:, None]                       # (P,W)
        avail = (L_UN == 0) & ~late
        emb = np.where(avail[:, :, None], L_EMB, np.nan).astype(np.float32)
        nt = np.where(late, np.nan, L_NT)
        un = (~avail).astype(np.int64)
        em = ((nt == 0) & avail).astype(np.int64)

        l_df = build_long_df_from_arrays(np.repeat(PIDS, N_WIN), np.tile(TRI_NAMES, P),
                                         emb.reshape(P * N_WIN, D), nt.reshape(-1), un.reshape(-1), em.reshape(-1))
        if CFG["save_raw_local"]:
            write_append(l_df, tbl_local(k))
        d_df = process_batch_rolling_delta_no_rampup(l_df, f"chunk {chunk_idx} wk{k:02d}", TRI_NAMES,
                                                     CFG["delta_n_back"], D)
        write_append(d_df, tbl_delta(k))
        del g_df, l_df, d_df, emb

    log_progress(chunk_idx, lo, hi, "done")
    total_done += P
    print(f"Chunk {chunk_idx + 1}/{n_chunks}: {P} patients x {len(CUTOFF_WEEKS)} cutoffs "
          f"({time.time()-t_chunk:.1f}s, {total_done/(time.time()-t_overall):.1f} patients/sec overall)")
    del G_EMB, G_VALID, G_SEQ, L_EMB, L_NT, L_UN, L_WEND
    gc.collect(); torch.cuda.empty_cache()

print(f"\nDone: {total_done} patients processed this run ({time.time()-t_overall:.1f}s)")

# excluded-patients audit table (same format as before)
excluded_df = pd.DataFrame({'patient_id': list(no_clinical_codes_ids),
                            'exclusion_reason': 'no_usable_clinical_codes'})
spark.createDataFrame(excluded_df).write.format("delta").mode("overwrite").saveAsTable(TBL_EXCLUDED)
print(f"Saved {len(excluded_df)} excluded patient IDs to {TBL_EXCLUDED}")

In [0]:
# Sanity summary: global rows per cutoff (should be non-decreasing as the cutoff advances)
for k in CUTOFF_WEEKS:
    print(f"wk{k:02d}: {spark.table(tbl_global(k)).count()} patients")